In [2]:
################################################################################
# 🧩 Analisis_Exp_Def_Full — Versión Simplificada y Corregida
# Fecha: 2025-11-10
# Autor: Ajustado por ChatGPT
################################################################################

import os
import gc
import pandas as pd
import numpy as np
from tqdm import tqdm

# =============================================================================
# 1️⃣ LIMPIEZA DE MEMORIA
# =============================================================================
def limpiar_memoria():
    """Libera memoria de objetos temporales"""
    gc.collect()
    print("🧹 Memoria limpiada correctamente.\n")

# =============================================================================
# 2️⃣ FUNCIÓN: CARGAR DICCIONARIO DE HOMOLOGACIÓN
# =============================================================================
def cargar_diccionario_homologacion(ruta_excel, hoja):
    """Carga el diccionario de homologación de campos."""
    try:
        df = pd.read_excel(ruta_excel, sheet_name=hoja)
        id_estandar = df.columns[0]  # Supongamos que la primera columna es el ID
        print(f"✅ Diccionario cargado con {len(df)} filas y ID estándar: {id_estandar}")
        return df, id_estandar
    except Exception as e:
        print(f"💥 Error cargando el diccionario: {e}")
        return None, None

# =============================================================================
# 3️⃣ FUNCIÓN: CARGAR ARCHIVOS PARQUET
# =============================================================================
def cargar_archivos_procesados(ruta_base="data/processed"):
    """Carga todos los archivos parquet válidos de defunciones en la carpeta."""
    print("\n📁 Buscando archivos en:", ruta_base)
    archivos = sorted([
        f for f in os.listdir(ruta_base)
        if f.startswith("defunciones_") and f.endswith("_procesado.parquet")
    ])
    
    if not archivos:
        print("⚠️ No se encontraron archivos para procesar.")
        return []

    lista_dataframes = []
    for i, archivo in enumerate(archivos, start=1):
        ruta = os.path.join(ruta_base, archivo)
        try:
            df = pd.read_parquet(ruta)
            print(f"✅ [{i:02d}] {archivo} — {df.shape[0]:,} filas, {df.shape[1]} columnas")
            lista_dataframes.append((archivo.replace(".parquet",""), df))
        except Exception as e:
            print(f"💥 Error al leer {archivo}: {e}")
    return lista_dataframes

# =============================================================================
# 4️⃣ FUNCIÓN: COMBINAR ARCHIVOS DE MANERA SEGURA Y OPTIMIZADA
# =============================================================================
def combinacion_simplificada_memoria(lista_dataframes, ruta_salida):
    """Combina múltiples DataFrames en memoria controlada."""
    print("\n🔄 Iniciando combinación simplificada...")
    chunks = []

    for nombre, df in lista_dataframes:
        if df is None or df.empty:
            print(f"⚠️ Saltando {nombre} (vacío o None)")
            continue
        
        # Limpieza de columnas tipo float/object inesperadas
        df = df.copy()
        for c in df.columns:
            if df[c].dtype == 'object':
                df[c] = df[c].astype(str)
        
        chunks.append(df)
        print(f"📦 Añadido: {nombre} — tamaño acumulado: {sum([len(x) for x in chunks]):,} filas")
        limpiar_memoria()

    chunks = [x for x in chunks if x is not None and not x.empty]
    if not chunks:
        raise ValueError("❌ No se pudo concatenar: todos los DataFrames están vacíos o None.")

    df_final = pd.concat(chunks, ignore_index=True)
    print(f"\n✅ Combinación completa: {df_final.shape[0]:,} filas, {df_final.shape[1]} columnas")

    # Guardado
    df_final.to_parquet(ruta_salida, index=False)
    print(f"💾 Archivo final guardado en: {ruta_salida}")

    return df_final

# =============================================================================
# 5️⃣ FUNCIÓN PRINCIPAL (VERSIÓN OPTIMIZADA Y ESTABLE)
# =============================================================================
def main():
    print("="*80)
    print("🚀 INICIANDO PROCESO DE COMBINACIÓN — MODO OPTIMIZADO")
    print("="*80)

    # 1️⃣ Crear carpeta de salida si no existe
    os.makedirs("data/processed", exist_ok=True)

    # 2️⃣ Cargar diccionario de homologación
    diccionario, id_estandar = cargar_diccionario_homologacion(
        "data/raw/Referenciales/Fuentes de Información Recolección Inicial.xlsx",
        "Campos Defunciones"
    )
    if diccionario is None:
        print("❌ No se pudo continuar sin el diccionario de homologación.")
        return

    # 3️⃣ Cargar archivos parquet procesados
    lista_dataframes = cargar_archivos_procesados("data/processed")
    if not lista_dataframes:
        print("❌ No hay archivos válidos para combinar.")
        return

    # 4️⃣ Combinar los archivos en modo optimizado (sin saturar memoria)
    try:
        print("\n📦 Iniciando combinación optimizada por partes...")
        combinacion_por_partes_y_guardado(
            lista_dataframes,
            "data/processed/defunciones_completo_final.parquet"
        )
        print("\n✅ PROCESO TERMINADO CON ÉXITO ✅")
        print("💾 Archivo final: data/processed/defunciones_completo_final.parquet")

        # Verificar tamaño final del archivo
        ruta = "data/processed/defunciones_completo_final.parquet"
        if os.path.exists(ruta):
            tamaño_mb = os.path.getsize(ruta) / (1024 * 1024)
            print(f"📊 Tamaño final del archivo: {tamaño_mb:.2f} MB")

    except Exception as e:
        print(f"\n💥 ERROR durante la combinación: {e}")
        import traceback
        traceback.print_exc()

    finally:
        limpiar_memoria()
        print("="*80)
        print("🏁 PROCESO FINALIZADO")
        print("="*80)


## =============================================================================
# 🔁 FUNCIÓN: COMBINACIÓN POR PARTES Y GUARDADO OPTIMIZADO (TODO STRING)
# =============================================================================
def combinacion_por_partes_y_guardado(lista_dataframes, ruta_salida_final):
    """
    Combina múltiples DataFrames grandes por partes sin consumir mucha RAM.
    Guarda cada parte como parquet temporal y al final los combina en uno solo.
    Convierte todas las columnas a string para evitar conflictos de tipo.
    """

    import pyarrow.parquet as pq
    import pyarrow as pa
    from pathlib import Path

    carpeta_temp = Path("data/combined_temp")
    carpeta_temp.mkdir(parents=True, exist_ok=True)

    total_filas = 0
    partes_guardadas = []

    print("\n🔄 Iniciando combinación por partes...\n")

    # ==========================================================
    # 🔹 PROCESAR Y GUARDAR CADA DATAFRAME POR SEPARADO
    # ==========================================================
    for i, (nombre, df) in enumerate(tqdm(lista_dataframes, desc="Procesando archivos")):
        if df is None or df.empty:
            print(f"⚠️ Saltando {nombre} (vacío o None)")
            continue

        # 🔸 Forzar todos los tipos de datos a string
        df = df.astype(str)

        # 🔸 Guardar cada parte como Parquet temporal
        nombre_temp = f"parte_{i+1:02d}_{nombre}.parquet"
        ruta_temp = carpeta_temp / nombre_temp
        df.to_parquet(ruta_temp, index=False)

        partes_guardadas.append(ruta_temp)
        total_filas += len(df)
        print(f"📦 Guardado: {ruta_temp} — filas acumuladas: {total_filas:,}")

        # 🔸 Liberar memoria
        del df
        limpiar_memoria()

    # ==========================================================
    # 🔹 COMBINAR LOS PARQUETS TEMPORALES EN UNO FINAL
    # ==========================================================
    print("\n🧩 Consolidando todos los archivos temporales en uno final...")

    tablas = []
    for archivo in tqdm(partes_guardadas, desc="Leyendo archivos parciales"):
        tabla = pq.read_table(archivo)
        tablas.append(tabla)

    # 🔸 Concatenar todas las tablas (ya sin conflicto de tipos)
    tabla_final = pa.concat_tables(tablas, promote=True)

    # 🔸 Guardar archivo final consolidado
    pq.write_table(tabla_final, ruta_salida_final)
    print(f"\n✅ Archivo final guardado en: {ruta_salida_final}")
    print(f"📊 Total de filas combinadas: {total_filas:,}")

    # ==========================================================
    # 🔹 LIMPIAR ARCHIVOS TEMPORALES (opcional)
    # ==========================================================
    try:
        for archivo in partes_guardadas:
            os.remove(archivo)
        print("🗑️ Archivos temporales eliminados correctamente.")
    except Exception as e:
        print(f"⚠️ No se pudieron eliminar archivos temporales: {e}")

    limpiar_memoria()



# =============================================================================
# 6️⃣ EJECUCIÓN
# =============================================================================
if __name__ == "__main__":
    main()


🚀 INICIANDO PROCESO DE COMBINACIÓN — MODO OPTIMIZADO
✅ Diccionario cargado con 1048575 filas y ID estándar: ID

📁 Buscando archivos en: data/processed
✅ [01] defunciones_1979_1991_procesado.parquet — 1,869,025 filas, 27 columnas
✅ [02] defunciones_1992_1996_procesado.parquet — 848,360 filas, 34 columnas
✅ [03] defunciones_1997_1997_procesado.parquet — 170,753 filas, 36 columnas
✅ [04] defunciones_1998_2007_procesado.parquet — 1,886,949 filas, 96 columnas
✅ [05] defunciones_2008_2011_procesado.parquet — 790,223 filas, 120 columnas
✅ [06] defunciones_2012_2013_procesado.parquet — 402,827 filas, 119 columnas
✅ [07] defunciones_2014_procesado.parquet — 210,051 filas, 114 columnas
✅ [08] defunciones_2015_procesado.parquet — 219,472 filas, 114 columnas
✅ [09] defunciones_2016_procesado.parquet — 223,078 filas, 114 columnas
✅ [10] defunciones_2017_procesado.parquet — 227,624 filas, 114 columnas
✅ [11] defunciones_2018_procesado.parquet — 236,932 filas, 114 columnas
✅ [12] defunciones_2019_pro

Procesando archivos:   6%|▌         | 1/17 [00:07<02:07,  7.99s/it]

📦 Guardado: data\combined_temp\parte_01_defunciones_1979_1991_procesado.parquet — filas acumuladas: 1,869,025
🧹 Memoria limpiada correctamente.



Procesando archivos:  12%|█▏        | 2/17 [00:12<01:30,  6.05s/it]

📦 Guardado: data\combined_temp\parte_02_defunciones_1992_1996_procesado.parquet — filas acumuladas: 2,717,385
🧹 Memoria limpiada correctamente.



...

📦 Guardado: data\combined_temp\parte_16_defunciones_2023_procesado.parquet — filas acumuladas: 8,549,253
🧹 Memoria limpiada correctamente.



Procesando archivos: 100%|██████████| 17/17 [02:55<00:00, 10.32s/it]


📦 Guardado: data\combined_temp\parte_17_defunciones_2024_procesado.parquet — filas acumuladas: 8,825,031
🧹 Memoria limpiada correctamente.


🧩 Consolidando todos los archivos temporales en uno final...


Leyendo archivos parciales: 100%|██████████| 17/17 [00:08<00:00,  1.98it/s]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_3192\1458242901.py:126: FutureWarning: promote has been superseded by promote_options='default'.
  combinacion_por_partes_y_guardado(



✅ Archivo final guardado en: data/processed/defunciones_completo_final.parquet
📊 Total de filas combinadas: 8,825,031
🗑️ Archivos temporales eliminados correctamente.
🧹 Memoria limpiada correctamente.


✅ PROCESO TERMINADO CON ÉXITO ✅
💾 Archivo final: data/processed/defunciones_completo_final.parquet
📊 Tamaño final del archivo: 262.37 MB
🧹 Memoria limpiada correctamente.

🏁 PROCESO FINALIZADO
